# Drag-and-drop photometry (real bandaid pipeline)

This notebook runs the **real [bandaid](https://github.com/mwcraig/bandaid)
photometry pipeline** on images drag-and-dropped into the JupyterLab file
browser — entirely in the browser, no helper service. It is
`watch_uploads.ipynb` (the plumbing testbed) with the simulated processing
step replaced by actual photometry: bandaid's `numpy-ballet` branch removes
jax from runtime, and both of the pipeline's network calls (the
Gaia-DR2-via-VizieR cone search and the HuggingFace weights download) are
CORS-clean, so neither needs a proxy.

**Recipe**

1. Run all cells in order. The weights cell downloads the ~39 MB Ballet CNN
   weights on first run and caches them in browser storage (IndexedDB), so
   later sessions skip the download. Another cell installs the folder-only
   drop guard. The last code cell before the light-curve section starts
   `watch()`, which polls `incoming/` once a second.
2. In the file browser, open `incoming/`, then drag a **folder** of Seestar
   FITS frames into it. (A folder is required: files inside a dropped folder
   upload one at a time, which keeps browser storage bounded; the drop guard
   rejects loose-file drops with a warning.)
3. The first frame that finishes uploading drives batch prep — source
   detection and FWHM on that frame, the Gaia cone search, contamination
   flagging, Bayer masks. Then every frame (including the first) is
   plate-solved, photometered, written to `results/<name>.star`, and deleted
   from `incoming/`. One line prints per frame: elapsed time, star count,
   FWHM, and the target star's counts.
4. When the loop exits, run the light-curve cell at the bottom.

**Stopping the loop**: create a file named `STOP` inside `incoming/` (file
browser: right-click → New File, rename to `STOP`), wait for the idle timeout
(default 120 s with no new files), or the notebook stop button if kernel
interrupt works in your deployment.


In [ ]:
# Drive-semantics smoke test: the kernel's filesystem is the JupyterLite
# contents drive, so listings/reads/deletes go through the live contents
# manager (IndexedDB), not a private kernel filesystem.
import os
import sys
import time

print("platform:", sys.platform)
print("cwd:     ", os.getcwd())
print("contents:", sorted(os.listdir(".")))

t0 = time.monotonic()
time.sleep(2)
print(f"time.sleep(2) took {time.monotonic() - t0:.2f}s")

# Manual checks (do these once per deployment, in the file browser UI):
# 1. Create a new text file at the drive root -> re-run this cell; it must
#    appear in the listing WITHOUT a kernel restart.
# 2. Run: open("kernel_test.txt", "w").write("x")  -> it must appear in the
#    file browser. Then os.remove("kernel_test.txt") -> it must disappear,
#    including from devtools > Application > IndexedDB > JupyterLite Storage.
# 3. Run time.sleep(30) and press the stop button to learn whether kernel
#    interrupt works here; the watch loop below does not depend on it.


In [ ]:
# Environment knobs first, then the bandaid imports.
import os
import sys

# astroquery imports keyring; there is no usable backend in wasm
os.environ.setdefault("PYTHON_KEYRING_BACKEND", "keyring.backends.null.Keyring")

# Negative-cache optional deps that are absent from this env: Python never
# caches a *failed* import, so astropy/photutils probe for these at call time
# and re-scan every sys.path directory on EVERY frame -- dozens of ~10 ms
# IndexedDB-drive stats, ~0.7 s/frame measured. sys.modules[name] = None
# makes the probe raise ImportError instantly instead; if one of these is
# ever added to the env, the successful import here caches it and it is
# used normally.
for _mod in ("gwcs", "bottleneck", "regions"):
    try:
        __import__(_mod)
    except ImportError:
        sys.modules[_mod] = None

import pyodide_http

pyodide_http.patch_all()  # requests -> browser fetch (VizieR + HuggingFace)

# The airmass AltAz transform would otherwise try to download IERS-A on first
# use; skip the fetch and accept degraded (sub-arcsecond) pointing accuracy,
# which is irrelevant at airmass precision.
from astropy.utils import iers

iers.conf.auto_download = False
iers.conf.iers_degraded_accuracy = "ignore"

import astropy.units as u
from astropy.coordinates import SkyCoord

from bandaid import (
    BatchPrepError,
    FrameError,
    PhotometryConfig,
    prepare_batch,
    write_starlist_set,
)
from bandaid.ballet_numpy import NumpyBallet
from bandaid.photometry import process_one_image
from bandaid.scripts import check_frame_consistency

print("bandaid imports OK")


In [ ]:
# Ballet CNN weights: download once (~39 MB), then reuse the copy cached in
# the JupyterLite drive (IndexedDB persists across sessions).
import numpy as np
import requests
from scipy.linalg.blas import sgemm
from scipy.special import expit

from bandaid.ballet_numpy import _max_pool_2x2_same

WEIGHTS_FILE = "ballet_weights.npz"
# Same repo/file/revision that bandaid's own download_weights() pins
# (bandaid.ballet_numpy) -- fetched with plain requests so huggingface_hub
# is never needed in the browser.
WEIGHTS_URL = ("https://huggingface.co/lgrcia/ballet/resolve/"
               "cfebd20240ce3fb694f6403a244f37f971e7780b/centroid_15x15.npz")

if os.path.exists(WEIGHTS_FILE):
    print(f"Using cached weights: {WEIGHTS_FILE} "
          f"({os.path.getsize(WEIGHTS_FILE) / 1e6:.1f} MB)")
else:
    resp = requests.get(WEIGHTS_URL, timeout=300)
    resp.raise_for_status()
    with open(WEIGHTS_FILE, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded {len(resp.content) / 1e6:.1f} MB -> {WEIGHTS_FILE}")


def _conv2d_same_sgemm(x, kernel, bias):
    """3x3 SAME conv as one im2col GEMM (same math as bandaid's einsum)."""
    n, h, w, c = x.shape
    o = kernel.shape[-1]
    xp = np.pad(x, ((0, 0), (1, 1), (1, 1), (0, 0)))
    win = np.lib.stride_tricks.sliding_window_view(xp, (3, 3), axis=(1, 2))
    # (n, h, w, c, 3, 3) -> (n*h*w, 3*3*c) with (i, j, c) column order,
    # matching the HWIO kernel's reshape to (9*c, o)
    cols = np.ascontiguousarray(win.transpose(0, 1, 2, 4, 5, 3))
    out = sgemm(1.0, cols.reshape(n * h * w, 9 * c), kernel.reshape(9 * c, o))
    return out.reshape(n, h, w, o) + bias


class SgemmBallet(NumpyBallet):
    """NumpyBallet with every matmul routed through scipy's BLAS.

    This kernel's numpy links no BLAS at all, so `@`/einsum fall back to
    scalar loops (~0.35 GFLOP/s measured here); scipy links the wasm
    openblas build (~8.3 GFLOP/s, 24x). Verified output-identical to
    NumpyBallet within float32 rounding (< 1e-6 px). Natively the two are
    the same speed, since numpy's `@` already is BLAS there.
    """

    def _forward(self, x):
        p = self.params
        x = x - x.min(axis=(1, 2, 3), keepdims=True)
        with np.errstate(invalid="ignore"):
            x = x / x.max(axis=(1, 2, 3), keepdims=True)
        for name in ("Conv_0", "Conv_1", "Conv_2"):
            x = _conv2d_same_sgemm(x, p[name]["kernel"], p[name]["bias"])
            x = np.maximum(x, 0.0)
            if name != "Conv_2":
                x = _max_pool_2x2_same(x)  # 15 -> 8, then 8 -> 4
        x = x.reshape(len(x), -1)
        x = expit(sgemm(1.0, x, p["Dense_0"]["kernel"]) + p["Dense_0"]["bias"])
        x = expit(sgemm(1.0, x, p["Dense_1"]["kernel"]) + p["Dense_1"]["bias"])
        return sgemm(1.0, x, p["Dense_2"]["kernel"]) + p["Dense_2"]["bias"]


cnn = SgemmBallet(model_file=WEIGHTS_FILE)


In [ ]:
# Editable run configuration.

# Merged into each frame's header-derived metadata; observer + site_* end up
# in the .star output, and site_lat/site_lon (+ optional site_elev) drive the
# airmass computation.
USER_META = {
    "observer": "LGEB",
    "site_elev": 1675,     # m
    "site_lon": -103.936,  # deg E
    "site_lat": 30.5952,   # deg N
}

# Defaults reproduce the reference Qatar-8 run made with `bandaid process`.
config = PhotometryConfig()

# Target star for the light-curve cell (Qatar-8, host of Qatar-8 b; ICRS
# per SIMBAD). The light curve uses the photometry star nearest these coords.
TARGET = SkyCoord(ra=157.41294 * u.deg, dec=70.52712 * u.deg)


In [ ]:
# Folder-only drop guard. The kernel runs in a web worker with no DOM access,
# but JupyterLab executes application/javascript outputs on the main thread,
# so this cell can install a page-level drop filter: a capture-phase listener
# that runs before JupyterLab's own drop handler and rejects any drop on the
# file browser that isn't purely folders (loose files upload in parallel and
# defeat the bounded-storage behavior). Entries are only inspectable during
# 'drop', not 'dragover', so the drag cursor still shows "copy" - the drop is
# refused on release, with a toast explaining why. Active only after this
# cell has run; harmless to re-run (guarded by a window flag).
from IPython.display import Javascript, display

FOLDER_ONLY_JS = """
(() => {
  if (window._folderOnlyDropGuard) { return; }
  window._folderOnlyDropGuard = true;
  const toast = (msg) => {
    const div = document.createElement('div');
    div.textContent = msg;
    Object.assign(div.style, {
      position: 'fixed', top: '12px', left: '50%',
      transform: 'translateX(-50%)', zIndex: 10000,
      background: 'var(--jp-warn-color1, #f57c00)', color: 'white',
      padding: '8px 16px', borderRadius: '4px',
      font: '13px var(--jp-ui-font-family, sans-serif)',
      boxShadow: '0 2px 8px rgba(0,0,0,0.3)',
    });
    document.body.appendChild(div);
    setTimeout(() => div.remove(), 5000);
  };
  document.addEventListener('drop', (ev) => {
    const onListing = ev.target instanceof Element && ev.target.closest('.jp-DirListing');
    if (!onListing) { return; }
    const items = Array.from(ev.dataTransfer?.items ?? []).filter((i) => i.kind === 'file');
    if (items.length === 0) { return; }  // not a native file drag
    const entries = items.map((i) => i.webkitGetAsEntry && i.webkitGetAsEntry());
    if (entries.every((e) => e && e.isDirectory)) { return; }  // all folders: allow
    ev.preventDefault();
    ev.stopImmediatePropagation();  // JupyterLab's upload handler never runs
    toast('Loose files rejected - drop a single folder of images instead.');
  }, true);  // capture phase: fires before the DirListing drop handler
})();
"""

display(Javascript(FOLDER_ONLY_JS))
print("Folder-only drop guard installed: dropping loose files on the file "
      "browser is now rejected with a warning; folders are accepted.")


In [ ]:
import shutil
import time
from pathlib import Path

from astropy.io import fits

WATCH_DIR = "incoming"
RESULTS_DIR = "results"
FITS_EXTS = (".fit", ".fits", ".fts")
MAX_RETRIES = 5  # consecutive fits-read failures before giving up on a file

# Batch state. prep is built from the first frame that finishes uploading
# (exactly what the bandaid CLI does with the first file of a batch); results
# accumulate per frame, in upload-arrival order, for the light-curve cell.
prep = None
target_idx = None
results = {}


def fits_candidates(root):
    """All FITS files under root, any depth (folder drops create subdirs)."""
    for dirpath, _dirnames, filenames in os.walk(root):
        for name in sorted(filenames):
            if name.lower().endswith(FITS_EXTS):
                yield os.path.join(dirpath, name)


def process_one(path):
    """Real photometry. Returns True if the frame was measured, False if it
    was skipped (recoverable per-frame failure); raises on unreadable files
    so the caller's retry logic can handle stalled uploads."""
    global prep, target_idx
    t0 = time.monotonic()
    # Photometer a MEMFS copy, not the contents-drive file: every syscall on
    # the drive is an ~10 ms IndexedDB round trip, and astropy's readers
    # stat/open the file dozens of times (~0.3 s/frame measured). One copy
    # to /tmp (kernel-private in-memory fs) turns that into a single read.
    local = os.path.join("/tmp", os.path.basename(path))
    shutil.copy(path, local)
    try:
        if prep is None:
            # BatchPrepError is fatal to *prep*, not to the run: leave prep as
            # None so the next frame retries it - a bad first frame doesn't
            # kill the whole session.
            try:
                prep = prepare_batch(local, cnn=cnn, config=config)
            except BatchPrepError as exc:
                print(f"[prep-fail] {path}: {exc}")
                return False
            target_idx = int(prep.photometry_coords.separation(TARGET).argmin())
            sep = prep.photometry_coords[target_idx].separation(TARGET).arcsec
            print(f"[prep {time.monotonic() - t0:5.1f}s] "
                  f"{len(prep.photometry_coords)} photometry stars; target is "
                  f"row {target_idx} ({sep:.1f} arcsec from TARGET)")
            t0 = time.monotonic()
        try:
            # path (not local) goes into the consistency check: it is only
            # attached to error messages, and the drive name is the one the
            # user recognizes.
            check_frame_consistency(path, fits.getheader(local), prep)
            by_filter = process_one_image(
                local,
                USER_META,
                prep.radecs,
                prep.cnn,
                prep.bayer_masks,
                config=prep.config,
                input_photometry_coords=prep.photometry_coords,
            )
            write_starlist_set(by_filter, Path(RESULTS_DIR) / (Path(path).stem + ".star"))
        except FrameError as exc:
            # Expected per-frame failures (WCS solve, too few stars, no usable
            # stars at write time): report and move on; the frame was fully
            # read, so the caller still deletes it - keeping it would wedge
            # the loop.
            print(f"[skip] {path}: {exc}")
            return False
    finally:
        os.remove(local)
    results[os.path.basename(path)] = by_filter
    l4 = by_filter["L4"]
    row = l4[target_idx]
    print(f"[done {time.monotonic() - t0:5.1f}s] {path}  "
          f"{len(l4)} stars  fwhm={l4.meta['fwhm']:.2f}px  "
          f"target L4: tot_count={row['tot_count']:.0f} snr={row['snr']:.1f}")
    return True


def prune_empty_dirs(root):
    """Remove emptied dropped-folder directories (bottom-up), never root itself."""
    for dirpath, dirnames, filenames in os.walk(root, topdown=False):
        if dirpath != root and not dirnames and not filenames:
            try:
                os.rmdir(dirpath)
            except OSError:
                pass


def watch(watch_dir=WATCH_DIR, poll=1.0, idle_timeout=120):
    os.makedirs(watch_dir, exist_ok=True)
    os.makedirs(RESULTS_DIR, exist_ok=True)
    stop_file = os.path.join(watch_dir, "STOP")
    done, failures, last_size = set(), {}, {}
    n_ok = n_skip = 0
    last_activity = time.monotonic()
    print(f"Watching {watch_dir!r}. Drop a folder of FITS files into it in the "
          f"file browser. To stop: create a file named STOP in {watch_dir!r} "
          f"(or wait {idle_timeout}s idle).")
    try:
        while True:
            if os.path.exists(stop_file):
                os.remove(stop_file)
                print("STOP file seen - exiting.")
                break
            for path in list(fits_candidates(watch_dir)):
                if path in done:
                    continue
                try:
                    size = os.path.getsize(path)
                except OSError:
                    continue  # vanished between walk and stat
                # Partial-upload guard: complete only when nonzero, unchanged
                # since last poll, and a whole number of 2880-byte FITS blocks
                # (mid-upload sizes are 1 MiB multiples, which fail this).
                complete = (size > 0
                            and last_size.get(path) == size
                            and size % 2880 == 0)
                last_size[path] = size
                last_activity = time.monotonic()  # new or growing file = activity
                if not complete:
                    continue
                try:
                    measured = process_one(path)
                except Exception as exc:
                    failures[path] = failures.get(path, 0) + 1
                    if failures[path] < MAX_RETRIES:
                        continue  # maybe a stalled upload resumed; retry next poll
                    print(f"[skip] {path}: unreadable after "
                          f"{MAX_RETRIES} tries ({exc}); left in place")
                    done.add(path)
                    continue
                done.add(path)
                if measured:
                    n_ok += 1
                else:
                    n_skip += 1
                os.remove(path)          # free IndexedDB immediately
                last_size.pop(path, None)
                last_activity = time.monotonic()
            prune_empty_dirs(watch_dir)
            if idle_timeout and time.monotonic() - last_activity > idle_timeout:
                print(f"No activity for {idle_timeout}s - exiting.")
                break
            time.sleep(poll)
    except KeyboardInterrupt:
        print("Interrupted - exiting.")
    print(f"Photometered {n_ok} frame(s); skipped {n_skip}; "
          f"{len(done) - n_ok - n_skip} unreadable.")


watch()


## Light curve

Run this after the watch loop exits. Row order in every frame's tables is
identical (all frames are measured at `prep.photometry_coords`), so the
target star found during batch prep is the same row everywhere. The plot
shows `tot_count`, normalized to its median, against time for the L4 channel
(the R+G+B sum) and the TG (green) channel.

The per-frame `.star` files in `results/` live in browser storage and persist
across sessions — delete the `results/` folder in the file browser when you
are done with them to reclaim space. The in-memory `results` dict is lost on
kernel restart; the `.star` files are the durable record.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

assert results, "No frames processed - run the watch cell (and drop files) first."


def series(filt):
    """(time, median-normalized tot_count) for the target star in one filter."""
    t = np.array([bf[filt]["time"][target_idx] for bf in results.values()])
    f = np.array([bf[filt]["tot_count"][target_idx] for bf in results.values()])
    order = np.argsort(t)
    t, f = t[order], f[order]
    return t, f / np.nanmedian(f)

t_l4, f_l4 = series("L4")
t_tg, f_tg = series("TG")
jd0 = np.floor(t_l4.min())

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(t_l4 - jd0, f_l4, s=14, label="L4 (R+G+B)", alpha=0.8)
ax.scatter(t_tg - jd0, f_tg, s=8, label="TG (green)", alpha=0.5)
ax.set_xlabel(f"JD $-$ {jd0:.0f}")
ax.set_ylabel("normalized tot_count")
ax.set_title(f"Qatar-8, {len(f_l4)} frames")
ax.legend()
ax.grid(alpha=0.3)
plt.show()
